# LongFlow — Gate Night 6 (the σ sweep: is there a sweet spot?)

Runtime: **L4 GPU**. ~45–60 min, ~$2. One arm, seven renders; every artifact
mirrors to Drive per run; reruns skip completed work. Pre-registered criteria:
`experiments/p1_flow_head/NOTES.md` (Gate Night 6 entry).

GN5 found σ=0.5 feedback noise rescues CONTENT (93% words, full 5-min render)
but kills IDENTITY (raspy whisper). This night maps the σ axis between 0 and
0.5 hunting a level that does both — plus one ramp schedule (H3). Everything
is held identical to GN5 Arm F (20K head, euler4, same script, seed 0) so the
seven renders and GN5's `f2_abl_noise` sit on one curve.

| cell | what | decides |
|---|---|---|
| 4 | **σ sweep** — base / 0.1 / 0.2 / 0.3 / 0.4 / 0.5 / ramp→0.5 | sweet spot vs tradeoff-curve vs cliff (H1/H2); ramp read (H3) |
| 6 | **FD backfill** — per-window Fréchet distance for all seven | the dose-response figure |
| 8 | Bundle | |


In [ ]:
# ===== COLD START (idempotent) — run me first, wait for READY =====
NOTEBOOK_VERSION = "GN6 v1.0 (2026-08-14): sigma sweep, GN5-comparable"
print(f"*** {NOTEBOOK_VERSION} ***")
%cd /content
!git clone -q https://github.com/vibevoice-community/VibeVoice.git 2>/dev/null || true
%cd /content/VibeVoice
!git checkout -q 07cb79fea
!pip install -q -e .

import torch
from vibevoice.modular.modeling_vibevoice_inference import (
    VibeVoiceForConditionalGenerationInference,
)
from vibevoice.processor.vibevoice_processor import VibeVoiceProcessor
MODEL_ID = "microsoft/VibeVoice-1.5B"
model = VibeVoiceForConditionalGenerationInference.from_pretrained(
    MODEL_ID, torch_dtype=torch.bfloat16, device_map="cuda"
)
processor = VibeVoiceProcessor.from_pretrained(MODEL_ID)
model.eval()

from google.colab import drive
drive.mount("/content/drive")
import glob, json, os, shutil, sys, time
import numpy as np
import soundfile as sf

if not os.path.exists("/content/LongFlow/src"):
    !git clone -q https://github.com/Josh-E-S/LongFlow.git /content/LongFlow || true
assert os.path.exists("/content/LongFlow/src"), "clone LongFlow or drag bundle"
sys.path.insert(0, "/content/LongFlow")
!cd /content/LongFlow && git log --oneline -1

from src.flow_head.cfm import euler_sample
from src.flow_head.integration import FlowHeadPatch
from src.flow_head.trainer import load_checkpoint

CKPT_DIR = "/content/drive/MyDrive/longflow_p1_ckpt"
TRAIN_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_cache"
EVAL_CACHE_DIR = "/content/drive/MyDrive/longflow_p1_evalcache"
GATE3_DIR = "/content/drive/MyDrive/longflow_gate3"
OUT = "/content/gate_night6"
DRIVE_OUT = "/content/drive/MyDrive/longflow_gate6"
os.makedirs(OUT, exist_ok=True)
os.makedirs(DRIVE_OUT, exist_ok=True)

head20, mean20, std20 = load_checkpoint(f"{CKPT_DIR}/full10k_20k.pt")
head20 = head20.to("cuda")

def gen_inputs(texts, prompt_lists):
    inputs = processor(text=texts, voice_samples=prompt_lists,
                       return_tensors="pt", padding=True)
    return {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in inputs.items()}

def save_wav(tag, wav, extra=None):
    sf.write(f"{OUT}/{tag}.wav", wav, 24000)
    shutil.copy(f"{OUT}/{tag}.wav", f"{DRIVE_OUT}/{tag}.wav")
    row = {"tag": tag, "audio_s": round(len(wav)/24000, 1)}
    if extra: row.update(extra)
    report["runs"].append(row)
    json.dump(report, open(f"{DRIVE_OUT}/gate_night6_report.json", "w"), indent=2)
    print(row, flush=True)

def done(tag):
    if os.path.exists(f"{DRIVE_OUT}/{tag}.wav"):
        print(f"skip {tag}", flush=True)
        return True
    return False

report = {"runs": [], "notebook_version": NOTEBOOK_VERSION}
print(f"READY — {NOTEBOOK_VERSION}")


In [ ]:
# shared inputs — same construction as GN5 so the script is IDENTICAL
# (same cache files, same slicing → same ~800 words); Drive listings can be
# transiently empty right after mount, retry before concluding anything
def drive_glob(pattern, tries=4, wait=15):
    for i in range(tries):
        hits = sorted(glob.glob(pattern))
        if hits:
            return hits
        print(f"empty listing for {pattern} — retry {i+1}/{tries} in {wait}s", flush=True)
        time.sleep(wait)
    raise RuntimeError(f"still empty after {tries} tries: {pattern} — check Drive mount/paths")

sents = []
for f in drive_glob(f"{TRAIN_CACHE_DIR}/*.pt")[-300:]:
    d = torch.load(f, weights_only=True)
    sents.append(d["text"].strip().rstrip(".") + ".")
pool = sents[:200]
prompts = drive_glob(f"{EVAL_CACHE_DIR}/*_prompt.wav")
P0 = prompts[0]

def turnscript(sentences, target=60, speaker=1):
    turns, cur, w = [], [], 0
    for s in sentences:
        cur.append(s); w += len(s.split())
        if w >= target:
            turns.append(f"Speaker {speaker}: " + " ".join(cur)); cur, w = [], 0
    if cur:
        turns.append(f"Speaker {speaker}: " + " ".join(cur))
    return "\n".join(turns) + "\n"

ABL_WORDS = []
w = 0
for s in pool:
    ABL_WORDS.append(s); w += len(s.split())
    if w >= 800: break
ABL_SCRIPT = turnscript(ABL_WORDS)
print("sweep script:", w, "words")
report["sweep_words"] = w
report["sweep_script"] = ABL_SCRIPT  # scorer computes WER vs this text


## 4. The σ sweep

GN5's noise intervention, σ parametrized. Mechanism byte-for-byte identical:
noise is added to the acoustic connector OUTPUT, gated on `patch.calls > 0`
(prompt encoding untouched — the v2 lesson), scaled by a running-std EMA.
`g6_sig000` is the base control re-rendered tonight; `g6_sig050` replicates
GN5 `f2_abl_noise` (consistency gate — if it doesn't reproduce, stop and
flag). `g6_ramp050` ramps σ linearly 0→0.5 over the first 450 frames (~60 s
@ 7.5 Hz), then holds — the H3 schedule probe. ~4–6 min per render.


In [ ]:
class NoiseIntervention:
    """GN5's ConnectorIntervention, noise mode only, sigma as a callable
    of the frame count so schedules are one lambda away."""
    def __init__(self, module, sigma_fn, active_fn=None):
        self.module, self.sigma_fn = module, sigma_fn
        self.active_fn = active_fn or (lambda: True)
        self.calls_fn = lambda: 0
        self.mu, self.var = None, None
        self.h = None
    def __enter__(self):
        def hook(mod, args, out):
            if not self.active_fn():   # untouched during prompt processing
                return out
            t = out[0] if isinstance(out, tuple) else out
            with torch.no_grad():
                if self.mu is None:
                    self.mu = t.mean().detach(); self.var = t.var().detach()
                else:
                    self.mu = 0.99*self.mu + 0.01*t.mean().detach()
                    self.var = 0.99*self.var + 0.01*t.var().detach()
                sigma = self.sigma_fn(self.calls_fn())
                if sigma <= 0:
                    return out
                new = t + torch.randn_like(t) * (sigma * self.var.sqrt())
            return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new
        self.h = self.module.register_forward_hook(hook)
        return self
    def __exit__(self, *exc):
        if self.h: self.h.remove()

CONDITIONS = [
    ("g6_sig000", lambda c: 0.0),
    ("g6_sig010", lambda c: 0.1),
    ("g6_sig020", lambda c: 0.2),
    ("g6_sig030", lambda c: 0.3),
    ("g6_sig040", lambda c: 0.4),
    ("g6_sig050", lambda c: 0.5),                      # GN5 f2_abl_noise replication
    ("g6_ramp050", lambda c: min(0.5, 0.5 * c / 450)), # 0 -> 0.5 over ~60 s, hold
]

for tag, sigma_fn in CONDITIONS:
    if done(tag):
        continue
    torch.manual_seed(0)
    with FlowHeadPatch(model, head20, mean20, std20, nfe=4, sway=0.0,
                       sampler=euler_sample) as patch, \
         NoiseIntervention(model.model.acoustic_connector, sigma_fn,
                           active_fn=lambda: patch.calls > 0) as nz, \
         torch.inference_mode():
        nz.calls_fn = lambda: patch.calls
        out = model.generate(**gen_inputs([ABL_SCRIPT], [[P0]]),
                             tokenizer=processor.tokenizer,
                             cfg_scale=1.3, max_new_tokens=3000)
    wav = out.speech_outputs[0].detach().float().cpu().numpy().squeeze()
    zs = torch.cat(patch.latents) if patch.latents else torch.zeros(1)
    save_wav(tag, wav, {"latent_std": round(float(zs.std()), 3),
                        "frames": patch.calls,
                        "sigma_final": round(sigma_fn(10**9), 2)})


## 6. FD backfill — the dose-response figure

Per-window Fréchet distance vs the GN3 teacher reference for all seven
renders (scalar std lied twice; FD is the collapse metric). Same code path
as GN5 Arm D. ~10 min.


In [ ]:
D_LAT = std20.numel()  # latent width from the training checkpoint

def unwrap(z):
    while isinstance(z, (tuple, list)):
        z = z[0]
    if hasattr(z, "sample"):
        z = z.sample() if callable(z.sample) else z.sample
    while isinstance(z, (tuple, list)):
        z = z[0]
    assert torch.is_tensor(z), f"could not unwrap encoder output: {type(z)}"
    return z

def encode_latents(path):
    x, sr = sf.read(path, dtype="float32")
    z_all = []
    step = 24000 * 30
    with torch.inference_mode():
        for i in range(0, len(x), step):
            seg = torch.from_numpy(x[i:i+step])[None, None].to("cuda", torch.bfloat16)
            z = unwrap(model.model.acoustic_tokenizer.encode(seg))
            while z.ndim > 2:
                z = z.squeeze(0)
            if z.shape[-1] != D_LAT and z.shape[0] == D_LAT:
                z = z.T
            assert z.shape[-1] == D_LAT, f"latent width {z.shape} vs ckpt {D_LAT}"
            z_all.append(z.float().cpu())
    out = torch.cat(z_all)
    print(f"  {os.path.basename(path)}: {len(out)} frames ({len(out)/7.5:.0f}s)", flush=True)
    return out

def fd_curve(z, ref_mu, ref_cov, win=75):  # 75 frames = 10 s @ 7.5 Hz
    import scipy.linalg
    out = []
    for i in range(0, len(z) - win, win):
        w = z[i:i+win].numpy()
        mu, cov = w.mean(0), np.cov(w.T)
        d = mu - ref_mu
        covmean = scipy.linalg.sqrtm(cov @ ref_cov)
        if np.iscomplexobj(covmean): covmean = covmean.real
        out.append(float(d @ d + np.trace(cov + ref_cov - 2*covmean)))
    return out

teacher_z = encode_latents(f"{GATE3_DIR}/t1_turnsplit_p0.wav")
half = len(teacher_z) // 2
ref_mu, ref_cov = teacher_z[:half].numpy().mean(0), np.cov(teacher_z[:half].numpy().T)

fd = {"teacher_self": fd_curve(teacher_z[half:], ref_mu, ref_cov)}
for tag, _ in CONDITIONS:
    p = f"{DRIVE_OUT}/{tag}.wav"
    if os.path.exists(p):
        fd[tag] = fd_curve(encode_latents(p), ref_mu, ref_cov)
report["fd_curves"] = fd
json.dump(report, open(f"{DRIVE_OUT}/gate_night6_report.json", "w"), indent=2)
for k, v in fd.items():
    print(f"{k}: first={v[0]:.1f} med={sorted(v)[len(v)//2]:.1f} last={v[-1]:.1f} n={len(v)}")


## 8. Bundle

`gate_night6_bundle.zip` (zips from the Drive mirror). Mac:
`unzip -o ~/Downloads/gate_night6_bundle.zip -d experiments/p1_flow_head/audio/gate_night6`
then `.venv/bin/python experiments/p1_flow_head/score_gate_night6.py`.


In [ ]:
import zipfile
json.dump(report, open(f"{DRIVE_OUT}/gate_night6_report.json", "w"), indent=2)
with zipfile.ZipFile("/content/gate_night6_bundle.zip", "w") as z:
    for f in glob.glob(f"{DRIVE_OUT}/*"):
        z.write(f, os.path.basename(f))
print("download /content/gate_night6_bundle.zip")
